<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_09%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. Ray Tracing의 기본 요소 3D 공간상의 물체를 2D 이미지로 렌더링하기 위해 가상의 카메라에서 픽셀마다 광선(Ray)을 쏘는 방식을 Ray Tracing이라고 합니다. 하나의 광선을 수학적으로 정의하기 위해 필요한 두 가지 핵심 벡터 요소는 무엇입니까?

(각각의 시작점과 뻗어나가는 성질을 의미합니다.)

답변 작성란:

1.광선의 원점 (ray origin)

2.광선의 방향 (ray direction)

- r(t) = o + td
  - o = 카메라 위치(원점)
  - d = 픽셀을 향해 뻗어나가는 방향
  - t = 광선 상의 거리

문항 2. NeRF의 출력값 NeRF(Neural Radiance Fields)는 3D 공간의 좌표 $(x, y, z)$와 보는 방향 $(\theta, \phi)$을 입력받아, 해당 지점의 광학적 특성을 나타내는 두 가지 값을 출력하도록 학습됩니다. 이 두 가지 출력값은 무엇입니까?

답변 작성란:

(시각적 정보): 색상 (Color, RGB)

(물체의 존재 여부/투명도와 관련): 밀도 (Density, sigma)

문항 3. 광선 생성하기 (Get Rays) 카메라의 내부 파라미터(초점거리 f)와 외부 파라미터(카메라 포즈 c2w)를 이용하여, 이미지의 각 픽셀을 통과하는 광선의 **방향(directions)** 과 **원점(rays_o)** 을 구하는 함수를 완성하세요.

In [ ]:
import numpy as np
import torch

def get_rays(H, W, f, c2w):
    # 1. 픽셀 그리드 생성 (i, j 좌표)
    i, j = torch.meshgrid(torch.arange(W, dtype=torch.float32),
                          torch.arange(H, dtype=torch.float32), indexing='xy')

    # TODO: 광선의 방향 벡터(dirs) 계산
    # 힌트: 픽셀 좌표를 정규화하고 카메라 좌표계로 변환. Z축 방향은 -1
    # 식: (x - W/2) / f, -(y - H/2) / f, -1
    dirs = torch.stack([(i - W * 0.5) / f,
                        -(j - H * 0.5) / f,
                        -torch.ones_like(i)
                       ], -1)

    # 2. 카메라 좌표계 -> 월드 좌표계 변환 (회전)
    rays_d = torch.sum(dirs[..., np.newaxis, :] * c2w[:3, :3], -1)

    # 3. 광선의 원점 (카메라 위치)
    rays_o = c2w[:3, -1].expand(rays_d.shape)

    return rays_o, rays_d

문항 4. Tiny NeRF 모델 정의 (MLP) 3D 좌표를 입력받아 색상과 밀도를 출력하는 간단한 다층 퍼셉트론(MLP) 모델을 정의합니다. 입력 차원(filter_size)에서 128개 노드로 가는 선형 층과, 마지막에 RGB(3) + Density(1)를 출력하는 층을 작성하세요.

In [ ]:
import torch.nn as nn

class VeryTinyNerfModel(nn.Module):
    def __init__(self, filter_size=128, num_encoding_functions=6):
        super(VeryTinyNerfModel, self).__init__()
        # 입력 차원: (x, y, z) 3개 * (2 * encoding_functions + 1)
        self.input_dim = 3 + 3 * 2 * num_encoding_functions

        # TODO: 첫 번째 선형 층 (Input -> 128)
        self.layer1 = nn.Linear(self.input_dim, 128)

        # 활성화 함수
        self.relu = nn.ReLU()

        # TODO: 두 번째 선형 층 (128 -> Output)
        # 출력 차원: RGB(3) + Density(1) = 4
        self.layer2 = nn.Linear(128, 4)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

문항 5. 계층적 샘플링 (Stratified Sampling) 볼륨 렌더링을 위해 광선 위에서 샘플링 포인트(z_vals)를 추출해야 합니다. 학습 시에는 오버피팅을 방지하고 공간을 고루 학습하기 위해 균일한 간격에 무작위 노이즈를 더해주는 방식을 사용합니다. 빈칸을 채워 샘플링 코드를 완성하세요.

In [ ]:
from torch._C import device
def render_rays(network_fn, rays_o, rays_d, near, far, n_samples, rand=False):
    # 1. 광선 위의 샘플링 지점 생성 (Linear spacing)
    t_vals = torch.linspace(0., 1., steps=n_samples)

    # 깊이(Depth) 값 계산: near에서 far 사이를 보간
    z_vals = near * (1.-t_vals) + far * (t_vals)

    if rand:
        # TODO: 계층적 샘플링 (Stratified Sampling)
        z_vals = z_vals + torch.rand(list(rays_o.shape[:-1]) + n_samples.to(device) * (far - near) / n_samples)
        mids = .5 * (z_vals[..., 1:] + z_vals[..., :-1])
        upper = torch.cat([mids, z_vals[..., -1:]], -1)
        lower = torch.cat([z_vals[..., :1], mids], -1)

        # 각 구간 내에서 무작위 위치 선택 (uniform noise)
        t_rand = torch.rand(z_vals.shape)
        z_vals = lower + (upper - lower) * t_rand

    # ... (이후 렌더링 로직 생략) ...
    return z_vals

문항 6. 학습 루프와 손실 함수 Tiny NeRF를 학습시키는 루프입니다. 모델이 예측한 이미지와 실제 타겟 이미지 간의 차이를 줄이기 위해 MSE(Mean Squared Error) Loss를 계산하는 코드를 작성하세요.

In [ ]:
# optimizer, model, data loader 등이 정의되어 있다고 가정

for i in range(1000):
    # 랜덤한 광선 배치 가져오기
    target_ray_values = target_img[target_indices] # (N, 3) 정답 RGB
    rays_o_batch = rays_o[target_indices]
    rays_d_batch = rays_d[target_indices]

    # 모델을 통해 렌더링 된 RGB 예측값
    rgb_predicted, _, _, _ = render_rays(model, rays_o_batch, rays_d_batch, near, far, N_samples)

    # TODO: 손실 함수 계산 (MSE Loss)
    # 예측값(rgb_predicted)과 정답(target_ray_values)의 차이의 제곱 평균
    loss = torch.mean((rgb_predicted - target_ray_values) ** 2)

    # 역전파 및 가중치 업데이트
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if i % 100 == 0:
        print(f"Iteration {i}, Loss: {loss.item()}")